In [3]:
import os
import mne
import pyedflib
import numpy as np
import scipy
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import jupyterlab
import yasa
import glob

# Configuration for interactive plots
%matplotlib widget

# Safe import of psutil with error handling in case it is not installed
try:    
    import psutil
except ImportError:    
    psutil = None

# Definition of the RAM check function
def print_ram(label=""):
    if psutil is None:
        print(f"{label} RAM: unavailable (psutil not installed)")
        return
        
    process = psutil.Process(os.getpid())
    mb = process.memory_info().rss / 1024 / 1024
    print(f"{label} RAM: {mb:.1f} MB")


print_ram("Initial state")

Initial state RAM: 473.0 MB


In [4]:

# Path to the folder containing your .edf files (change to your actual path)
data_folder = "/mnt/c/Users/olivi/Desktop/___EEG/CZD_sampleData/CZD_sampleData/eeg_edf"

# Retrieve a list of paths to all .edf files in that folder
edf_files = glob.glob(f"{data_folder}/*.edf")

# Display the found files and their total count
print(f"Found {len(edf_files)} EDF files.")
for file in edf_files:
    print(file)

Found 3 EDF files.
/mnt/c/Users/olivi/Desktop/___EEG/CZD_sampleData/CZD_sampleData/eeg_edf/2023-07-26{6591E658-D0E0-49FE-BDE4-FC101EACF45D} Data.edf
/mnt/c/Users/olivi/Desktop/___EEG/CZD_sampleData/CZD_sampleData/eeg_edf/2023-07-27{891F8634-3CC8-4FEE-B1D3-64394DC86F7A} Data.edf
/mnt/c/Users/olivi/Desktop/___EEG/CZD_sampleData/CZD_sampleData/eeg_edf/2023-07-27{D29553B9-1220-4DFC-8456-F712EE2281C8} Data.edf


In [ ]:
def create_bipolar(raw_input):

    electrodes = {}

    for ch in raw_input.ch_names:
        if not ch[-1].isdigit():
            continue

        electrode = ch.rstrip("0123456789")

        if electrode not in electrodes:
            electrodes[electrode] = []

        electrodes[electrode].append(ch)

    anode = []
    cathode = []
    bipolar_names = []

    for electrode in electrodes:
        # Sort channels by their actual number (integer value)
        channels = sorted(
            electrodes[electrode],
            key=lambda x: int(x[len(electrode):])
        )

        for i in range(len(channels) - 1):
            # Extract numerical numbers for the current and next channel
            current_num = int(channels[i][len(electrode):])
            next_num = int(channels[i + 1][len(electrode):])
            
            # CONDITION: Create a pair only if they are adjacent (the difference is exactly 1)
            if next_num - current_num == 1:
                anode.append(channels[i])
                cathode.append(channels[i + 1])
                bipolar_names.append(f"{channels[i]}-{channels[i + 1]}")
            else:
                # Optional: you can add a print statement here, e.g., print(f"Skipped gap: {channels[i]} to {channels[i+1]}")
                continue

    raw_bipolar = mne.set_bipolar_reference(
        raw_input,
        anode=anode,
        cathode=cathode,
        ch_name=bipolar_names,
        drop_refs=False,
        copy=True
    )
 
    return raw_bipolar, bipolar_names

In [ ]:
#trzeba napewno sprawdzic czy sciezki do zapisania sa okay!!!!



# Loop through each EDF file one by one
for edf_file in edf_files:
    # Get the base name of the file without the extension
    file_base_name = os.path.splitext(os.path.basename(edf_file))[0]
    print(f"\nProcessing file: {file_base_name}")
    
    # 1. Create a dedicated output folder for this specific file
    output_dir = f"SOZ/{file_base_name}"
    os.makedirs(output_dir, exist_ok=True)
    
    # 2. Load metadata (preload=False keeps RAM free)
    raw = mne.io.read_raw_edf(edf_file, preload=False)

    # 3. Extract and search markers from annotations
    annotations = pd.DataFrame({
        "onset_s": raw.annotations.onset,
        "duration_s": raw.annotations.duration,
        "description": raw.annotations.description
    })

    # Save the full annotations table inside the patient's main directory
    annotations.to_csv(
        f"{output_dir}/inventory_table_annotations.csv",
        index=False
    )

    # Find rows where the description contains the word "NAPAD"
    seizure_rows = annotations[annotations["description"].str.contains("NAPAD", na=False)]

    # Check if any seizure event was found in this file
    if seizure_rows.empty:
        print(f"No seizure ('NAPAD') found in {file_base_name}. Skipping to next file.")
        continue  # Skip the rest of the loop for this file

    print(f"Found {len(seizure_rows)} seizure event(s) ('NAPAD') in this file.")

    # Loop through each detected seizure in this specific file
    for index, row in enumerate(seizure_rows.itertuples(), start=1):
        onset = row.onset_s
        print(f" -> Processing Seizure #{index} at onset time: {onset:.1f} seconds")

        # Define the 40-second window centered around this specific onset (-10s to +30s)
        tmin = onset - 10
        tmax = onset + 30
        print(f"    Calculated window: {tmin:.1f}s to {tmax:.1f}s")

        # Create unique subfolders for this specific seizure's output files
        seizure_output_dir = f"{output_dir}/seizure_{index}"
        os.makedirs(seizure_output_dir, exist_ok=True)
        os.makedirs(f"{seizure_output_dir}/spectrograms", exist_ok=True)

        # 4. Crop and load the 40-second window data into RAM
        raw_cropped = raw.copy().crop(tmin=tmin, tmax=tmax)
        raw_cropped.load_data()

        # 5. Filter the data
        raw_filtered = raw_cropped.copy()
        raw_filtered.notch_filter(freqs=[50, 100, 150], fir_design='firwin')
        raw_filtered.filter(l_freq=1.0, h_freq=70.0, fir_design='firwin')

        # 6. Create bipolar montage from the filtered data
        raw_bipolar_filtered, bipolar_names = create_bipolar(raw_filtered)
        total_channels = len(bipolar_names)
        print(f"    Created bipolar channels: {total_channels}")

        # 7. Plot and save the EEG wave trace
        fig_filtered = raw_bipolar_filtered.plot(
            picks=bipolar_names,
            n_channels=15,
            duration=10.0,
            scalings="auto",
            title=f"FILTERED Bipolar trace - Seizure {index} ({total_channels} chs)",
            show=False
        )
        fig_filtered.savefig(f"{seizure_output_dir}/bipolar_filtered_trace.png", dpi=300)
        plt.close(fig_filtered)  # Free memory up immediately

        # 8. Compute, plot, and save the PSD graph
        print(f"    Computing PSD for Seizure #{index}...")
        psd = raw_bipolar_filtered.compute_psd(
            fmin=0,
            fmax=100,
            picks=bipolar_names
        )

        fig_psd = psd.plot(
            average=False,
            show=False
        )
        fig_psd.suptitle(
            f"PSD - bipolar channels - Seizure {index}",
            fontsize=14
        )
        fig_psd.savefig(
            f"{seizure_output_dir}/psd_bipolar.png",
            dpi=300,
            bbox_inches="tight"    
        )   
        plt.close(fig_psd)  # Free memory up immediately

        # 9. Compute and save spectrograms for every bipolar channel tego nie jestem pewna poronac z analizsys_long_file
        sfreq = raw_bipolar_filtered.info["sfreq"]
        print(f"    Generating spectrograms for {total_channels} channels...")
        
        for channel_for_spectrogram in bipolar_names:
            signal = raw_bipolar_filtered.get_data(picks=[channel_for_spectrogram])[0]

            f, t, Sxx = scipy.signal.spectrogram(
                signal,
                fs=sfreq,
                nperseg=512,
                noverlap=256
            )

            fig_spec = plt.figure(figsize=(10, 5))
            plt.pcolormesh(
                t,
                f,
                10 * np.log10(Sxx),
                shading="gouraud"
            )
            plt.ylim(0, 100)
            plt.xlabel("Time [s]")
            plt.ylabel("Frequency [Hz]")
            plt.title(f"Spectrogram - {channel_for_spectrogram} (Seizure {index})")
            plt.colorbar(label="Power [dB]")
            plt.tight_layout()

            # Save the spectrogram into the specific subfolder
            fig_spec.savefig(
                f"{seizure_output_dir}/spectrograms/spectrogram_{channel_for_spectrogram}.png",
                dpi=300
            )
            plt.close(fig_spec)  # Free memory up immediately

        # 10. Clear loop-specific variables to prevent RAM bloat
        del raw_cropped, raw_filtered, raw_bipolar_filtered, psd
        plt.close('all')

    print(f"Finished processing all {len(seizure_rows)} seizures for: {file_base_name}")